# IP Catcher - Generador de APK
## Ejecuta esta celda para generar el APK

In [ ]:
# Instalar dependencias
!apt update -qq
!apt install -qq -y wget unzip openjdk-17-jdk git
!pip install -qq buildozer kivy

# Configurar JAVA
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

# Descargar Android SDK
!wget -q https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip -O cmdline-tools.zip
!mkdir -p android-sdk/cmdline-tools
!unzip -q cmdline-tools.zip -d android-sdk/cmdline-tools
!mv android-sdk/cmdline-tools/cmdline-tools android-sdk/cmdline-tools/latest

# Configurar variables
os.environ['ANDROIDSDK'] = '/content/android-sdk'
os.environ['PATH'] = os.environ['PATH'] + ':/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools'

# Aceptar licencias e instalar componentes
!yes | /content/android-sdk/cmdline-tools/latest/bin/sdkmanager --licenses
!sdkmanager "platform-tools" "platforms;android-34" "build-tools;34.0.0" -q

print('SDK instalado!')

In [ ]:
# Crear archivos del proyecto
!mkdir -p app

# Main.py
with open('app/main.py', 'w') as f:
    f.write('''
from kivy.app import App
from kivy.uix.label import Label
from kivy.uix.textinput import TextInput
from kivy.uix.button import Button
from kivy.uix.boxlayout import BoxLayout
import socket, requests, platform
from datetime import datetime

class IPcatcherApp(App):
    def build(self):
        layout = BoxLayout(orientation='vertical', padding=20, spacing=10)
        self.status = Label(text='[+] IP Catcher [+]', font_size=20, size_hint_y=None, height=50)
        layout.add_widget(self.status)
        self.url_input = TextInput(hint_text='URL del servidor', multiline=False, size_hint_y=None, height=50)
        layout.add_widget(self.url_input)
        self.info_label = Label(text='', font_size=12)
        layout.add_widget(self.info_label)
        btn = Button(text='CAPTURAR Y ENVIAR', size_hint_y=None, height=60)
        btn.bind(on_press=self.send_data)
        layout.add_widget(btn)
        return layout

    def get_info(self):
        try:
            local_ip = socket.gethostbyname(socket.gethostname())
        except:
            local_ip = 'No disponible'
        try:
            public_ip = requests.get('https://api.ipify.org', timeout=5).text
        except:
            public_ip = 'No disponible'
        info = [
            '=== CAPTURA ===',
            f'IP Local: {local_ip}',
            f'IP Publica: {public_ip}',
            f'Android: {platform.platform()}',
            f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
            '='*20
        ]
        return '\n'.join(info)

    def send_data(self, instance):
        url = self.url_input.text.strip()
        if not url:
            self.status.text = '[!] Ingresa URL'
            return
        self.status.text = '[...] Enviando...'
        try:
            info = self.get_info()
            requests.post(f'{url}/save', data={'info': info, 'type': 'android'}, timeout=10)
            self.status.text = '[OK] Enviado!'
            self.info_label.text = info
        except Exception as e:
            self.status.text = '[ERROR]'
            self.info_label.text = str(e)[:50]

if __name__ == '__main__':
    IPcatcherApp().run()
''')

# buildozer.spec
spec = '''[app]
title = IP Catcher
package.name = ipcatcher
package.domain = com.tool
source.dir = .
version = 1.0
requirements = python3,kivy,requests
orientation = portrait
android.permissions = INTERNET,ACCESS_NETWORK_STATE
'''

with open('buildozer.spec', 'w') as f:
    f.write(spec)

print('Archivos creados!')

In [ ]:
# Compilar APK
!cd /content && buildozer android debug 2>&1 | tail -20

# Copiar APK
import shutil
import os

if os.path.exists('bin'):
    for f in os.listdir('bin'):
        if f.endswith('.apk'):
            shutil.copy(f'bin/{f}', 'IP_Catcher.apk')
            print(f'APK creado: {f}')
else:
    print('Esperando...')
    import time
    time.sleep(60)
    if os.path.exists('bin'):
        for f in os.listdir('bin'):
            if f.endswith('.apk'):
                shutil.copy(f'bin/{f}', 'IP_Catcher.apk')
                print(f'APK creado: {f}')

In [ ]:
# Descargar APK
from google.colab import files
files.download('IP_Catcher.apk')